<a href="https://colab.research.google.com/github/yoshiagent/mathModel-colab/blob/main/notebooks/DC_Motor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 直流馬達的建模與模擬 DC Motor

對照網頁：[直流馬達](https://yoshiagent.github.io/mathModel/DCM.html)（無負載、有負載、等效轉矩與慣量、伺服馬達選用）

| 原檔 | 內容 | 本筆記本 |
|---|---|---|
| `DCMotorData.m`、`DCMotorTf.mdl`、`DCMotorTf1.mdl` | 電樞方程式＋機械方程式、化簡後的轉移函數 | 第 1 節 |
| `DCMotorDataSS.m`、`DCMotorSS.mdl` | 狀態空間 | 第 1 節 |
| 方塊圖化簡（Why?） | $G/(1+GH)$ 的推導 | 第 2 節 |
| `DCMotorDataMaxon.m`、`DCMotorMaxon1.mdl` | Maxon 110117 型錄參數、無載轉速與電流、三個時間常數 | 第 3 節 |
| `DCMotorTfTL.mdl` | 外加負載轉矩 $T_L$ | 第 4 節 |
| `DCMotorTfTLRotD.m`、`DCMotorTfTLRot.mdl`、`DCMotorTfTLRot1.mdl` | 經減速齒輪帶動轉動負載：兩種模型等價 | 第 5 節 |
| Servo Motor Selection | 加減速轉矩、RMS 轉矩與電流（Baldor 範例） | 第 6 節 |

Simulink 模型的參數與設定（步階時間、模擬時間）都取自原 .mdl 檔。只用 Colab 內建的 numpy、scipy、sympy、plotly。

In [1]:
import numpy as np
import sympy as sp
from scipy import signal
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def show(fig, title, xlab=None, ylab=None, h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()


def motor_rhs(p, ea, TL=lambda t: 0.0, Jeff=None, Beff=None):
    """狀態 (theta, omega, i)：L di/dt = ea - R i - Ke w；J dw/dt = Kt i - B w - TL"""
    J = p["Jm"] if Jeff is None else Jeff
    B = p["B"] if Beff is None else Beff

    def f(t, x):
        th, w, i = x
        return [w, (p["Kt"] * i - B * w - TL(t)) / J, (ea(t) - p["R"] * i - p["Ke"] * w) / p["L"]]
    return f


def step_at(t0, amp):
    return lambda t: amp if t >= t0 else 0.0

## 1. 無負載的直流馬達（`DCMotorData.m`）

$$e_a = L\frac{di_a}{dt} + Ri_a + K_E\omega_m,\qquad J_m\frac{d\omega_m}{dt} + B_m\omega_m = K_T i_a$$

$$\frac{\Omega_m(s)}{E_a(s)} = \frac{K_T}{LJ_m s^2 + (RJ_m + LB_m)s + RB_m + K_EK_T}$$

```matlab
L=0.5; B=0.1; R=1; Jm=0.01; Ke=0.01; Kt=0.01;     % DCMotorData.m
```
`DCMotorTf.mdl`（電氣與機械兩個方塊＋反電動勢回授）、`DCMotorTf1.mdl`（化簡成單一轉移函數）、`DCMotorSS.mdl`（狀態空間）
三個模型都是：Step 在 $t = 1$ s 由 0 跳到 1 V，模擬 10 s；原講義的 Display 顯示轉速 0.0999 rad/s、轉角 0.8392 rad。

In [2]:
P1 = dict(L=0.5, B=0.1, R=1.0, Jm=0.01, Ke=0.01, Kt=0.01)
L, B, R, Jm, Ke, Kt = (P1[k] for k in ("L", "B", "R", "Jm", "Ke", "Kt"))
t = np.linspace(0, 10, 2001)

# (a) DCMotorTf.mdl：方塊圖＝微分方程式
sol = solve_ivp(motor_rhs(P1, step_at(1, 1.0)), (0, 10), [0, 0, 0], t_eval=t, max_step=0.01, rtol=1e-10, atol=1e-12)
# (b) DCMotorTf1.mdl：化簡後的轉移函數 Kt/(L Jm s^2 + (R Jm + L B) s + R B + Ke Kt)，再積分得轉角
num, den = [Kt], [L * Jm, R * Jm + L * B, R * B + Ke * Kt]
u = (t >= 1).astype(float)
_, w_tf, _ = signal.lsim(signal.lti(num, den), u, t, interp=False)
th_tf = np.concatenate([[0], np.cumsum((w_tf[1:] + w_tf[:-1]) / 2 * np.diff(t))])
# (c) DCMotorDataSS.m：x = [theta_dot; i_a]
A = np.array([[-B / Jm, Kt / Jm], [-Ke / L, -R / L]])
Bs = np.array([[0], [1 / L]])
_, w_ss, _ = signal.lsim(signal.StateSpace(A, Bs, np.array([[1, 0]]), np.array([[0]])), u, t, interp=False)

print(f"steady speed Kt/(R B + Ke Kt) = {Kt / (R * B + Ke * Kt):.4f} rad/s   (Display1: 0.0999)")
print(f"theta(10): ODE {sol.y[0, -1]:.4f}, transfer function {th_tf[-1]:.4f}   (Display: 0.8392)")
print(f"max |speed| difference: ODE vs TF {np.max(np.abs(sol.y[1] - w_tf)):.1e}, TF vs SS {np.max(np.abs(w_tf - w_ss)):.1e}")
print("poles:", np.round(np.roots(den), 4), " -> time constants", np.round(-1 / np.roots(den), 4), "s")
fig = make_subplots(rows=1, cols=2, subplot_titles=("speed (rad/s)", "angle (rad)"))
fig.add_scatter(x=t, y=np.round(sol.y[1], 6), name="speed: ODE (DCMotorTf.mdl)", row=1, col=1)
fig.add_scatter(x=t, y=np.round(w_ss, 6), name="speed: state space", line=dict(dash="dash"), row=1, col=1)
fig.add_scatter(x=t, y=np.round(sol.y[0], 6), name="angle", row=1, col=2)
fig.update_xaxes(title="t (s)")
show(fig, "DC motor without load: 1 V step at t = 1 s")

steady speed Kt/(R B + Ke Kt) = 0.0999 rad/s   (Display1: 0.0999)
theta(10): ODE 0.8392, transfer function 0.8392   (Display: 0.8392)
max |speed| difference: ODE vs TF 9.8e-12, TF vs SS 2.1e-16
poles: [-9.9975 -2.0025]  -> time constants [0.1    0.4994] s


若電感 $L$ 很小（電氣時間常數遠小於機械時間常數）可以忽略：$\dfrac{\Omega_m}{E_a} \approx \dfrac{K_T}{RJ_m s + RB_m + K_EK_T}$。
這組參數 $L = 0.5$ H 不算小（$\tau_e = L/R = 0.5$ s，$\tau_{mech} = J_m/B_m = 0.1$ s），忽略 $L$ 的模型反而差很多：

In [3]:
_, w_noL, _ = signal.lsim(signal.lti([Kt], [R * Jm, R * B + Ke * Kt]), u, t, interp=False)
print(f"tau_e = L/R = {L / R:.2f} s, tau_mech = Jm/B = {Jm / B:.2f} s;  max speed error when L is neglected: "
      f"{np.max(np.abs(w_noL - w_tf)) / w_tf[-1] * 100:.0f} % of the final speed")

tau_e = L/R = 0.50 s, tau_mech = Jm/B = 0.10 s;  max speed error when L is neglected: 67 % of the final speed


## 2. 方塊圖化簡：$\dfrac{Y}{R} = \dfrac{G}{1+GH}$

用 sympy 由方塊圖的兩條關係式 $E = R - HY$、$Y = GE$ 解出閉迴路轉移函數，再代入馬達：
$G = \dfrac{1}{Ls+R}\,K_T\,\dfrac{1}{J_m s + B_m}$、$H = K_E$。

In [4]:
s, Gs, Hs, Rs, Es, Ys = sp.symbols("s G H R E Y")
solY = sp.solve([sp.Eq(Es, Rs - Hs * Ys), sp.Eq(Ys, Gs * Es)], [Es, Ys])[Ys]
print("Y/R =", sp.simplify(solY / Rs))
Ls_, Rr_, Kt_, Ke_, Jm_, Bm_ = sp.symbols("L R K_T K_E J_m B_m", positive=True)
G_motor = Kt_ / ((Ls_ * s + Rr_) * (Jm_ * s + Bm_))
cl = sp.factor(sp.simplify(G_motor / (1 + G_motor * Ke_)))
print("Omega/Ea =", cl)
print("expanded denominator:", sp.collect(sp.expand(sp.denom(cl)), s))

Y/R = G/(G*H + 1)
Omega/Ea = K_T/(B_m*L*s + B_m*R + J_m*L*s**2 + J_m*R*s + K_E*K_T)
expanded denominator: B_m*R + J_m*L*s**2 + K_E*K_T + s*(B_m*L + J_m*R)


## 3. Maxon 110117 馬達（`DCMotorDataMaxon.m`、`DCMotorMaxon1.mdl`）

型錄：無載轉速 9630 rpm、無載電流 30 mA、$K_T = K_E = 5.9$ mNm/A、$R = 1.71\ \Omega$、$L = 0.11$ mH、$J_m = 3.88$ g·cm²。
摩擦係數由無載狀態估計：$B = K_T I_0/\omega_0$。`DCMotorMaxon1.mdl`：6 V 在 $t = 1$ s 加上，模擬 10 s。

In [5]:
w0 = 9630 * 2 * np.pi / 60
Bmx = 5.9e-3 * 30e-3 / w0
PM = dict(L=0.11e-3, B=Bmx, R=1.71, Jm=3.88e-7, Ke=5.9e-3, Kt=5.9e-3)
print(f"omega0 = {w0:.1f} rad/s, B = Kt I0 / omega0 = {Bmx:.4e} N m s/rad")
t3 = np.concatenate([np.linspace(0, 1, 11)[:-1], np.linspace(1, 1.2, 4001), np.linspace(1.2, 10, 881)[1:]])
sol3 = solve_ivp(motor_rhs(PM, step_at(1, 6.0)), (0, 10), [0, 0, 0], t_eval=t3, method="LSODA", rtol=1e-10, atol=1e-12, max_step=1e-4)
w_ss3 = 6 * PM["Kt"] / (PM["R"] * PM["B"] + PM["Ke"] * PM["Kt"])
print(f"no-load speed: {sol3.y[1, -1]:.1f} rad/s = {sol3.y[1, -1] * 60 / (2 * np.pi):.0f} rpm (catalog 9630), formula {w_ss3:.1f};  "
      f"no-load current: {sol3.y[2, -1] * 1000:.2f} mA (catalog 30);  angle(10) = {sol3.y[0, -1]:.0f} rad (Display: 9055)")
tau_e, tau_m = PM["L"] / PM["R"], PM["Jm"] / PM["B"]
tau_em = PM["R"] * PM["Jm"] / (PM["R"] * PM["B"] + PM["Ke"] * PM["Kt"])
k63 = np.argmax(sol3.y[1] >= 0.632 * w_ss3)
print(f"tau_e = {tau_e * 1000:.4f} ms, tau_mech = {tau_m:.2f} s, tau_em = {tau_em * 1000:.2f} ms (catalog 'mechanical time constant' 19 ms);"
      f"  simulated time to 63%: {(t3[k63] - 1) * 1000:.1f} ms")
k = (t3 >= 0.98) & (t3 <= 1.12)
fig = make_subplots(rows=1, cols=2, subplot_titles=("speed (rad/s)", "current (A)"))
fig.add_scatter(x=np.round(t3[k], 5), y=np.round(sol3.y[1][k], 3), name="speed", row=1, col=1)
fig.add_scatter(x=np.round(t3[k], 5), y=np.round(sol3.y[2][k], 5), name="current", row=1, col=2)
fig.update_xaxes(title="t (s)")
show(fig, "Maxon 110117: 6 V step at t = 1 s (zoom 0.98-1.12 s)")

omega0 = 1008.5 rad/s, B = Kt I0 / omega0 = 1.7552e-07 N m s/rad


no-load speed: 1008.3 rad/s = 9628 rpm (catalog 9630), formula 1008.3;  no-load current: 29.99 mA (catalog 30);  angle(10) = 9055 rad (Display: 9055)
tau_e = 0.0643 ms, tau_mech = 2.21 s, tau_em = 18.90 ms (catalog 'mechanical time constant' 19 ms);  simulated time to 63%: 18.9 ms


## 4. 外加負載轉矩（`DCMotorTfTL.mdl`）

$T(s) = (J_m s + B_m)\Omega_m(s) + T_L(s)$。模型設定：6 V 在 $t = 0$ 加上，負載轉矩 0.01 N·m 在 $t = 1$ s 加上，模擬 2 s（Maxon 參數）。
原講義 Display：轉速 521.2 rad/s、轉角 1520 rad。轉速下降量 $\Delta\omega = \dfrac{T_L R}{RB_m + K_EK_T}$。

補充：負載時電流約 1.71 A，超過型錄的最大連續電流 0.84 A（0.01 N·m 也大於最大連續轉矩 4.96 mN·m），這只是示範用的數字，實際上這顆小馬達不能長時間這樣運轉。

In [6]:
t4 = np.concatenate([np.linspace(0, 0.2, 2001), np.linspace(0.2, 1, 81)[1:-1], np.linspace(1, 1.2, 2001), np.linspace(1.2, 2, 81)[1:]])
sol4 = solve_ivp(motor_rhs(PM, step_at(0, 6.0), TL=step_at(1, 0.01)), (0, 2), [0, 0, 0], t_eval=t4, method="LSODA", rtol=1e-10, atol=1e-12, max_step=1e-4)
dw = 0.01 * PM["R"] / (PM["R"] * PM["B"] + PM["Ke"] * PM["Kt"])
print(f"speed with load: {sol4.y[1, -1]:.1f} rad/s (Display1 521.2); drop {w_ss3 - sol4.y[1, -1]:.1f} = TL R/(R B + Ke Kt) = {dw:.1f};  "
      f"angle(2) = {sol4.y[0, -1]:.0f} rad (Display 1520);  current with load = {sol4.y[2, -1]:.3f} A")
fig = make_subplots(rows=1, cols=2, subplot_titles=("speed (rad/s)", "current (A)"))
fig.add_scatter(x=np.round(t4, 5), y=np.round(sol4.y[1], 3), name="speed", row=1, col=1)
fig.add_scatter(x=np.round(t4, 5), y=np.round(sol4.y[2], 5), name="current", row=1, col=2)
fig.update_xaxes(title="t (s)")
show(fig, "DCMotorTfTL.mdl: 6 V at t = 0, load torque 0.01 N m at t = 1 s")

speed with load: 521.2 rad/s (Display1 521.2); drop 487.0 = TL R/(R B + Ke Kt) = 487.0;  angle(2) = 1520 rad (Display 1520);  current with load = 1.710 A


## 5. 經齒輪帶動的轉動負載（`DCMotorTfTLRotD.m`、`DCMotorTfTLRot.mdl`、`DCMotorTfTLRot1.mdl`）

```matlab
Ke=0.06; Kt=0.06; L=0.02; R=1.2; N1=16; N2=32;          % DCMotorTfTLRotD.m（ICST-5 馬達）
Jm=6.2*10^(-4); Bm=10^(-4); JL=8.2*10^(-4); BL=0.15*10^(-5);
```
負載轉矩 $T_L = J_L\left(\frac{N_1}{N_2}\right)^2\dot\omega_m + B_L\left(\frac{N_1}{N_2}\right)^2\omega_m$。
`DCMotorTfTLRot.mdl` 把它畫成 du/dt 回授（框內的迴路），`DCMotorTfTLRot1.mdl` 直接用等效的
$\dfrac{1}{J_T s + B_T}$。兩個模型：1 V 在 $t = 0$ 加上、模擬 2 s，輸出負載端轉角 $\theta_L = \frac{N_1}{N_2}\theta_m$。

In [7]:
P5 = dict(Ke=0.06, Kt=0.06, L=0.02, R=1.2, Jm=6.2e-4, B=1e-4)
n = 16 / 32
JL, BL = 8.2e-4, 0.15e-5
JT, BT = P5["Jm"] + n ** 2 * JL, P5["B"] + n ** 2 * BL
t5 = np.linspace(0, 2, 2001)
# (a) TfTLRot.mdl：TL 由 du/dt 回授算出 → 代數迴路 J_m dw = Kt i - Bm w - (JL n^2 dw + BL n^2 w)
s_a = solve_ivp(lambda tt, x: [x[1], (P5["Kt"] * x[2] - P5["B"] * x[1] - BL * n ** 2 * x[1]) / (P5["Jm"] + JL * n ** 2),
                               (1.0 - P5["R"] * x[2] - P5["Ke"] * x[1]) / P5["L"]], (0, 2), [0, 0, 0], t_eval=t5, rtol=1e-10, atol=1e-12)
# (b) TfTLRot1.mdl：等效轉動慣量與摩擦
s_b = solve_ivp(motor_rhs(P5, step_at(0, 1.0), Jeff=JT, Beff=BT), (0, 2), [0, 0, 0], t_eval=t5, rtol=1e-10, atol=1e-12)
print(f"J_T = {JT:.4e}, B_T = {BT:.4e};  load angle theta_L(2) = {n * s_a.y[0, -1]:.4f} / {n * s_b.y[0, -1]:.4f} rad;  "
      f"max difference {np.max(np.abs(s_a.y[0] - s_b.y[0])):.1e}")
print(f"motor speed -> {s_b.y[1, -1]:.3f} rad/s (Kt/(R B_T + Ke Kt) = {P5['Kt'] / (P5['R'] * BT + P5['Ke'] * P5['Kt']):.3f}), load speed {n * s_b.y[1, -1]:.3f} rad/s")
_, _, s_n = None, None, solve_ivp(motor_rhs(P5, step_at(0, 1.0)), (0, 2), [0, 0, 0], t_eval=t5, rtol=1e-10, atol=1e-12)
fig = go.Figure()
fig.add_scatter(x=t5, y=np.round(n * s_b.y[1], 5), name="load speed (with load inertia, J_T)")
fig.add_scatter(x=t5, y=np.round(n * s_n.y[1], 5), name="motor without load x N1/N2", line=dict(dash="dot"))
show(fig, "Geared rotational load: equivalent inertia J_T = Jm + (N1/N2)^2 JL slows the start-up", "t (s)", "rad/s")

J_T = 8.2500e-04, B_T = 1.0038e-04;  load angle theta_L(2) = 13.9777 / 13.9777 rad;  max difference 1.1e-14
motor speed -> 16.121 rad/s (Kt/(R B_T + Ke Kt) = 16.127), load speed 8.061 rad/s


## 6. 伺服馬達選用（Baldor BSM 範例，英制單位）

馬達 $J_m = 0.0037$ lb-in-s²、$K_t = 4.8$ lb-in/A；負載 200 lb、滾珠螺桿 $A = 0.2$ in/rev、螺桿慣量 0.00313 lb-in-s²、摩擦轉矩 0.95 lb-in。
運動曲線：加速 0.12 s → 等速 2000 rpm 0.12 s → 減速 0.12 s → 停 0.3 s。

In [8]:
w_run = 2000 * 2 * np.pi / 60
acc = 209 / 0.12                                   # 原講義把 209.4 取成 209 rad/s
J1 = 200 / 386 * (0.2 / (2 * np.pi)) ** 2
Jt = J1 + 0.00313 + 0.0037
Tf = 0.95
Tacc, Tdec, Trun = Jt * acc + Tf, -Jt * acc + Tf, Tf
Trms = np.sqrt((Tacc ** 2 * 0.12 + Trun ** 2 * 0.12 + Tdec ** 2 * 0.12) / (0.12 + 0.12 + 0.12 + 0.3))
print(f"speed {w_run:.1f} rad/s, accel {acc:.1f} rad/s^2, J1 = {J1:.5f}, Jt = {Jt:.5f} lb-in-s^2")
print(f"Tacc = {Tacc:.2f}, Tdec = {Tdec:.2f}, Trun = {Trun:.2f} lb-in;  T_RMS^2 = {Trms ** 2:.2f}, T_RMS = {Trms:.2f} lb-in")
Ta2, Td2 = (0.00052 + 0.00313 + 0.0037) * 1741.6 + 0.95, -(0.00052 + 0.00313 + 0.0037) * 1741.6 + 0.95   # 原講義的取位
Tr2 = np.sqrt((Ta2 ** 2 + 0.95 ** 2 + Td2 ** 2) * 0.12 / 0.66)
print(f"with the slide's rounded numbers: Tacc = {Ta2:.2f}, Tdec = {Td2:.2f}, T_RMS^2 = {Tr2 ** 2:.2f}, T_RMS = {Tr2:.2f} lb-in (slide: 59.86, 7.73)")
print(f"I_acc = {Tacc / 4.8:.2f} A, I_RMS = {Trms / 4.8:.2f} A  (continuous stall torque 14.4 lb-in > T_RMS: OK)")
tt = np.array([0, 0.12, 0.24, 0.36, 0.66])
fig = make_subplots(rows=1, cols=2, subplot_titles=("speed profile (rpm)", "torque (lb-in)"))
fig.add_scatter(x=tt, y=[0, 2000, 2000, 0, 0], name="speed", row=1, col=1)
fig.add_scatter(x=[0, 0.12, 0.12, 0.24, 0.24, 0.36, 0.36, 0.66], y=np.round([Tacc, Tacc, Trun, Trun, Tdec, Tdec, 0, 0], 3),
                name="torque", line_shape="hv", row=1, col=2)
fig.add_hline(y=Trms, line_dash="dash", annotation_text=f"T_RMS = {Trms:.2f}", row=1, col=2)
fig.update_xaxes(title="t (s)")
show(fig, "Servo motor selection: move profile and required torque")

speed 209.4 rad/s, accel 1741.7 rad/s^2, J1 = 0.00052, Jt = 0.00735 lb-in-s^2
Tacc = 13.76, Tdec = -11.86, Trun = 0.95 lb-in;  T_RMS^2 = 60.16, T_RMS = 7.76 lb-in
with the slide's rounded numbers: Tacc = 13.75, Tdec = -11.85, T_RMS^2 = 60.08, T_RMS = 7.75 lb-in (slide: 59.86, 7.73)
I_acc = 2.87 A, I_RMS = 1.62 A  (continuous stall torque 14.4 lb-in > T_RMS: OK)
